# **LLM Pipeline to Process Extracted Dataset (Phase 1)**

In [ ]:
!pip install pandas

**Locate extracted dataset**

In [46]:
from pathlib import Path, PurePosixPath

def find_dataset(dirname: str, search_root=None):
    root = search_root or Path.cwd()

    direct = root / dirname

    return direct if direct.is_dir() else None

In [2]:
DATASET_NAME = "extracted_dataset_1792212375"
DATASET_PATH = find_dataset(DATASET_NAME)

#  Define DISEASES by scanning the directories in DATASET_PATH
DISEASES = sorted([item.name for item in DATASET_PATH.iterdir() if item.is_dir()])

# **Rough Processing**

**Helper functions to rename and retrieve cases**

In [26]:
import pandas as pd

def rename_cases(disease_name):
    disease_folder = DATASET_PATH / disease_name
    old_file = disease_folder / "cases.csv"
    new_file = disease_folder / f"{disease_name}_cases.csv"

    if old_file.exists():
        old_file.rename(new_file)
    else:
        print(f"File {old_file} does not exist or has already been renamed.")

In [ ]:
# Rename once
for disease in DISEASES:
    rename_cases(disease_name=disease)

**Plumbing functions for set intersections, subtractions and retrieval**

In [ ]:
def retrieve_cases(disease_name: str, only_id: bool = False):
    disease_folder = DATASET_PATH / disease_name
    csv_path = disease_folder / f"{disease_name}_cases.csv"

    if only_id:
        # Load only the 'case_id' column and immediately convert to a Python list
        return pd.read_csv(csv_path, usecols=['case_id'])['case_id'].tolist()

    # Otherwise, return the full pandas DataFrame
    return pd.read_csv(csv_path)

In [27]:
def process_subtractions(target: list, amount: list[list]) -> list:
    """
    Subtracts multiple lists of items from a target list.
    target: A single list of case_ids
    amount: A list of lists of case_ids
    """
    result_set = set(target)
    for sub_list in amount:
        result_set -= set(sub_list)
    return list(result_set)

def process_intersections(lists_to_intersect: list[list]) -> list:
    # Takes a list of lists and returns a single list containing only the elements present in every provided list.
    if not lists_to_intersect:
        return []

    # Start with a set of the first list
    common_elements = set(lists_to_intersect[0])

    # Intersect with the sets of the remaining lists
    for lst in lists_to_intersect[1:]:
        common_elements &= set(lst)

    return list(common_elements)

**Plumbing function to handle `image_metadata.json`**

In [67]:
def iter_image_metadata(disease_name: str, relative: bool = False):
    """
    Plumbing generator that iterates through image_metadata.json.
    Yields: (raw_json_line, case_id, resolved_absolute_path)
    """
    metadata_path = DATASET_PATH / disease_name / "image_metadata.json"

    if not metadata_path.exists():
        return

    with open(metadata_path, 'r') as file:
        for line in file:
            line_str = line.strip()
            if not line_str:
                continue

            record = json.loads(line_str)
            case_id = record.get("case_id")
            original_path = record.get("file_path", "")

            resolved_path = None
            if original_path and original_path.startswith("medical_datasets/"):
                # Drop the root directory name and resolve against DATASET_PATH
                relative_parts = PurePosixPath(original_path).parts[1:]

                if relative:
                    resolved_path = "/".join(relative_parts)
                else:
                    resolved_path = DATASET_PATH.joinpath(*relative_parts)

            yield line_str, case_id, resolved_path

def get_image_paths_dict(disease_name: str, relative: bool = False) -> dict:
    """
    Returns a dictionary mapping case_id to a list of resolved image paths.
    """
    case_id_to_images = {}
    for _, case_id, resolved_path in iter_image_metadata(disease_name, relative=relative):
        if resolved_path:
            # Group all resolved paths by case_id
            case_id_to_images.setdefault(case_id, []).append(str(resolved_path))

    return case_id_to_images

**Plumbing function to delete specific cases (via `case_id`)**

In [49]:
import json
def delete_cases(disease_name: str, case_ids_to_delete: list | str | int):
    """
    Atomically deletes cases and their associated images.
    Step 1: Removes image files and their entries from image_metadata.json.
    Step 2: Removes the cases from the disease-specific cases.csv.
    """
    if not isinstance(case_ids_to_delete, list):
        case_ids_to_delete = [case_ids_to_delete]

    if not case_ids_to_delete:
        return  # Nothing to do

    delete_set = set(case_ids_to_delete)

    # ==========================================
    # PART 1: Delete Images and JSON Metadata
    # ==========================================
    metadata_path = DATASET_PATH / disease_name / "image_metadata.json"

    if metadata_path.exists():
        # Keep track of lines to keep
        lines_to_keep = []

        # Iterate over every line in JSON
        for raw_line, case_id, img_path in iter_image_metadata(disease_name):
            if case_id in delete_set and img_path:
                try:
                    # Attempt to delete the physical image file
                    if img_path.exists() and img_path.is_file():
                        img_path.unlink()
                        print(f"Successfully deleted image: {img_path}")
                        continue  # Skip appending this line (metadata removed)
                    else:
                        print(f"Image not found at: {img_path}. Metadata retained.")
                except Exception as e:
                    print(f"Error deleting image {img_path}: {e}. Metadata retained.")

            # If not deleted, keep the exact original line format
            lines_to_keep.append(raw_line + "\n")

        # Write the filtered metadata back
        with open(metadata_path, 'w') as file:
            file.writelines(lines_to_keep)
    else:
        print(f"Metadata file not found at {metadata_path}. Skipping image deletion phase.")

    # ==========================================
    # PART 2: Delete Records from Master CSV
    # ==========================================
    csv_path = DATASET_PATH / disease_name / f"{disease_name}_cases.csv"

    if not csv_path.exists():
        print(f"Cases CSV file not found at {csv_path}")
        return

    # Use our retrieve_cases helper
    cases_df = retrieve_cases(disease_name)

    if 'case_id' not in cases_df.columns:
        print(f"'case_id' column missing in {csv_path}")
        return

    initial_row_count = len(cases_df)

    # Use our process_subtractions helper to filter the IDs
    all_ids = cases_df['case_id'].tolist()
    ids_to_keep = process_subtractions(target=all_ids, amount=[case_ids_to_delete])

    # Filter the dataframe to only include the kept IDs
    df_filtered = cases_df[cases_df['case_id'].isin(ids_to_keep)]

    final_row_count = len(df_filtered)

    # Save changes to disk
    if initial_row_count > final_row_count:
        df_filtered.to_csv(csv_path, index=False)
        print(f"Successfully deleted {initial_row_count - final_row_count} case record(s) from CSV for: {disease_name}")
    else:
        print(f"No matching CSV entries found to delete for: {disease_name}")

### **Porcelain Function 1: Eliminate cases with ambiguous demographic**

In [50]:
def delete_unknown_demographics(disease_name: str) -> list[dict]:
    """
    Retrieves cases for a disease, finds cases where age is NaN or
    gender is NaN/'Unknown', deletes them atomically (including images),
    and returns a list of dicts containing the deleted 'case_id' and 'pmc_id'.
    """
    #  Retrieve the master dataframe
    df = retrieve_cases(disease_name)

    # Safety check in case the dataframe is empty or missing columns
    if df.empty or 'age' not in df.columns or 'gender' not in df.columns:
        print(f"Skipping demographics check for {disease_name}: Required columns missing.")
        return []

    # Create a mask for missing age or missing/unknown gender
    mask = (
        df['age'].isna() |
        df['gender'].isna() |
        (df['gender'].astype(str).str.strip().str.lower() == 'unknown')
    )

    flagged_df = df[mask]

    if flagged_df.empty:
        return []

    # (Checking for both 'pmcid' and 'pmc_id' just in case column names vary across datasets)
    if 'pmcid' in flagged_df.columns:
        return_df = flagged_df[['case_id', 'pmcid']].rename(columns={'pmcid': 'pmc_id'})
    elif 'pmc_id' in flagged_df.columns:
        return_df = flagged_df[['case_id', 'pmc_id']]
    else:
        return_df = flagged_df[['case_id']]  # Fallback

    deleted_records = return_df.to_dict(orient='records')

    # Trigger the atomic deletion of these cases
    case_ids_to_delete = flagged_df['case_id'].tolist()
    delete_cases(disease_name, case_ids_to_delete)

    return deleted_records

In [51]:
for disease in DISEASES:
    try:
        delete_unknown_demographics(disease)
    except Exception as e:
        print(f"An error occurred while processing '{disease}': {e}")
print("\nDeleted all cases with ambiguous demographic!")


Deleted all cases with ambiguous demographic


# **LLM Processing (Remember to put the gpt and claude folders in first)**

**Plumbing function to categorize cases based on LLM agreement (both agree, both disagree, conflict)**

In [31]:
def evaluate_cases(disease: str) -> tuple[list, list, list]:
    """
    Calculates the passed, failed, and conflict case IDs.
    Returns: passed_both_ids, failed_both_ids, conflict_ids
    """
    disease_dir = DATASET_PATH / disease

    # Read only 'case_id' column to save memory during subtraction
    master_ids = retrieve_cases(disease, only_id=True)

    passed_id_lists = []
    failed_id_lists = []

    # Search every subfolder inside disease_dir except folders named 'image' or 'images'
    for subfolder in disease_dir.iterdir():
        if subfolder.is_dir() and subfolder.name.lower() not in ["image", "images"]:
            for file in subfolder.glob("*.csv"):
                filename_lower = file.name.lower()

                if "passed" in filename_lower:
                    ids = pd.read_csv(file, usecols=['case_id'])['case_id'].tolist()
                    passed_id_lists.append(ids)
                elif "failed" in filename_lower:
                    ids = pd.read_csv(file, usecols=['case_id'])['case_id'].tolist()
                    failed_id_lists.append(ids)

    if not passed_id_lists or not failed_id_lists:
        print(f"Skipping {disease}: Insufficient passed/failed files found in model directories.")
        return [], [], []

    # Process intersections using the helper function
    passed_both_ids = process_intersections(passed_id_lists)
    failed_both_ids = process_intersections(failed_id_lists)

    # Process conflicts using the helper function
    conflict_ids = process_subtractions(target=master_ids, amount=[passed_both_ids, failed_both_ids])

    return passed_both_ids, failed_both_ids, conflict_ids

### **Porcelain Function 2: Export .csv(s) based on agreement category**

In [44]:
def export_evaluation_csvs(disease: str):
    """
    Filters the master cases dataframe using the evaluated lists, exports them to .csv files,
    and generates a formatted text file with queries for each conflicting case.
    """
    disease_dir = DATASET_PATH / disease

    # Load the full master list to retain all original columns for the exported CSVs
    cases_df = retrieve_cases(disease)

    # Generate a .csv file for each agreement category
    passed_ids, failed_ids, conflict_ids = evaluate_cases(disease)

    if passed_ids:
        cases_passed = cases_df[cases_df['case_id'].isin(passed_ids)]
        cases_passed.to_csv(disease_dir / f"{disease}_passed_both.csv", index=False)

    if failed_ids:
        cases_failed = cases_df[cases_df['case_id'].isin(failed_ids)]
        cases_failed.to_csv(disease_dir / f"{disease}_failed_both.csv", index=False)

    if conflict_ids:
        cases_conflict = cases_df[cases_df['case_id'].isin(conflict_ids)]

        # Export the conflict CSV
        cases_conflict.to_csv(disease_dir / f"{disease}_cases_conflict.csv", index=False)

        # Generate the text file with formatted queries directly from the DataFrame in memory
        txt_path = disease_dir / f"{disease}_conflict_queries.txt"
        with open(txt_path, 'w', encoding='utf-8') as txt_file:
            for i, row in enumerate(cases_conflict.itertuples(), start=1):
                case_id = getattr(row, 'case_id', 'Unknown ID')
                case_text = getattr(row, 'case_text', 'No text available')

                block = (
                    f"Example {i}:\n"
                    f"Case ID: {case_id}\n"
                    f"Case Report: {case_text}\n"
                    f'Question: "Does this patient have {disease}?"\n'
                    "\n"
                )
                txt_file.write(block)

        print(f"Successfully formatted {len(cases_conflict)} queries into {txt_path.name}")

    print(f"Exported CSVs for {disease}: {len(passed_ids)} passed both, {len(failed_ids)} failed both, {len(conflict_ids)} conflicts.")

In [ ]:
for disease in DISEASES:
        print(f"\n--- Processing {disease} ---")
        export_evaluation_csvs(disease)
print("\nExport completed!")

### **Porcelain Function 3: Delete cases irrelevant to label disease**

In [36]:
def delete_irrelevant_files(disease_name: str):
    """
    Deletes cases that failed in both models, as well as defective cases
    (conflicts that failed the triple check). Atomically removes CSV entries and images.
    """
    # Mini-function: encapsulated triple check logic
    def _triple_check(conflict_ids: list) -> list:
        if not conflict_ids:
            return []

        disease_dir = DATASET_PATH / disease_name
        triple_files = list(disease_dir.glob("triple*.csv"))

        if not triple_files:
            print(f"Skipping triple check for {disease_name}: No 'triple' file found. Assuming all conflicts are defective.")
            return conflict_ids

        triple_file = triple_files[0]
        triple_ids = pd.read_csv(triple_file, usecols=['case_id'])['case_id'].tolist()

        # Defective = Conflict - Triple
        return process_subtractions(target=conflict_ids, amount=[triple_ids])

    # Evaluate the state of the directories
    passed_both_ids, failed_both_ids, conflict_ids = evaluate_cases(disease_name)

    # Determine defective cases after third check
    defective_ids = _triple_check(conflict_ids)

    # Combine failed cases and defective cases into a single deletion list
    case_ids_to_delete = list(set(failed_both_ids) | set(defective_ids))

    # 3. PLUMBING OUT: Flush the bad data
    if case_ids_to_delete:
        delete_cases(disease_name, case_ids_to_delete)
    else:
        print(f"No irrelevant files to delete for {disease_name}.")

In [ ]:
for disease in DISEASES:
        print(f"\n--- Processing {disease} ---")
        delete_irrelevant_files(disease)
print("\nDeleted all irrelevant cases!")

# **Final Processing (Split into train and test)**

**Obtain secret from Colab Secrets or API key**

In [103]:
import os
from pathlib import Path

def get_secret(secret_name: str, env_file: str = "public.env", set_env_var: bool = True) -> str:
    """
    Retrieves a secret/API key by attempting:
    1. Google Colab Secrets (google.colab.userdata)
    2. A local .env file (via python-dotenv)

    Args:
        secret_name: The key name (e.g., 'HF_TOKEN', 'OPENAI_API_KEY').
        env_file: Path to the .env file fallback (default: 'public.env').
        set_env_var: If True, automatically sets os.environ[secret_name] = secret.

    Returns:
        The secret value as a stripped string.

    Raises:
        KeyError: If secret cannot be retrieved from any source.
    """
    # 1. Try Google Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
        if val and val.strip():
            secret = val.strip()
            if set_env_var:
                os.environ[secret_name] = secret
            return secret
    except Exception:
        pass  # Not in Colab, secret missing, or permission denied

    # 2. Fall back to .env file
    env_path = Path.cwd() / env_file
    if env_path.is_file():
        try:
            from dotenv import dotenv_values
            val = dotenv_values(env_path).get(secret_name)
            if val and val.strip():
                secret = val.strip()
                if set_env_var:
                    os.environ[secret_name] = secret
                return secret
        except ImportError:
            raise ImportError("python-dotenv is required to read .env files. Run: pip install python-dotenv")

    # Raise clear error if both methods fail
    raise KeyError(
        f"Could not load '{secret_name}' from Colab Secrets or '{env_file}'. "
        "Ensure the secret exists and is populated."
    )

**Use normalized disease names (Disease Ontology)**

In [107]:
import requests
def search_umls(query_string, api_key=get_secret("UMLS_API_KEY")):
    """Searches UMLS for a specific string and returns basic results."""

    # The base URL for the UMLS REST API search endpoint
    url = "https://uts-ws.nlm.nih.gov/rest/search/current"

    # The parameters for the GET request
    params = {
        "string": query_string,
        "apiKey": api_key,
        "pageNumber": 1 # Optional: defaults to 1
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # Raise an exception for bad status codes

        data = response.json()

        # Extract the results from the JSON response
        results = data.get("result", {}).get("results", [])

        if not results:
            print(f"No results found for '{query_string}'")
            return

        return results[0]['name']

    except requests.exceptions.RequestException as e:
        print(f"Error communicating with the UMLS API: {e}")

In [ ]:
# Normalize disease names and save to this list
NORMALIZED_DISEASES = []

for disease in DISEASES:
    normalized_name = search_umls(disease)

    NORMALIZED_DISEASES.append({disease : normalized_name})

### **Porcelain Function 4: Build big dataset**

In [116]:
def build_combined_dataset() -> pd.DataFrame:
    """
    Iterates over all diseases, extracts cases and images, and builds
    a massive combined cases.csv at the root of DATASET_PATH.
    """
    all_rows = []

    # Convert list of dicts into a fast O(1) lookup dictionary
    normalized_map = {k: v for item in NORMALIZED_DISEASES for k, v in item.items()}

    # Iterate through all valid disease folders in DATASET_PATH
    for disease in DISEASES:

        # Retrieve cases of that disease
        disease_df = retrieve_cases(disease)

        if disease_df.empty:
            continue

        # Retrieve the parsed image dictionary
        image_paths_dict = get_image_paths_dict(disease, relative=False)

        # Format the core dataframe
        rows = pd.DataFrame({
            "case_id": disease_df["case_id"],
            "case_text": disease_df["case_text"],
            "disease": normalized_map.get(disease, disease),
        })

        # Add length of case text (in words)
        rows["case_len"] = rows["case_text"].fillna("").astype(str).apply(
            lambda text: len(text.split())
        )

        # Map the dictionary to create the image payloads
        rows["images"] = rows["case_id"].map(
            lambda cid: [
                {"image_path": p, "image_id": PurePosixPath(p).name}
                for p in image_paths_dict[cid]
            ] if image_paths_dict.get(cid) else None
        )

        all_rows.append(rows)

    # Combine everything and export to root
    combined_df = pd.concat(all_rows, ignore_index=True)

    output_csv = DATASET_PATH / "cases.csv"
    combined_df.to_csv(output_csv, index=False)

    print(f"Successfully built massive dataset at {output_csv} with {len(combined_df)} total cases.")
    return combined_df

In [117]:
all_cases_df = build_combined_dataset()

Successfully built massive dataset at /home/tokuden/VGU_WS26_ClinicalProject_BHTBH/BHTBH/extracted_dataset_1792212375/cases.csv with 1740 total cases.


### **Porcelain Function 5: Split big dataset into train and test using Round Robin**

In [118]:
import numpy as np

def separate_naive_round_robin(df: pd.DataFrame, group_col: str = "disease", seed: int = 42, stop_on_first_exhaustion: bool = True):
    """
    Splits a dataframe by taking 1 test case and 4 train cases per group in a round-robin fashion.
    To prevent majority classes from skewing the dataset, the loop terminates when the first group runs out of enough cases to complete a pass.
    """
    rng = np.random.default_rng(seed)

    # Create a dictionary of shuffled indices for each disease pool
    pool_indices = {}
    for group_val, group in df.groupby(group_col):
        shuffled_idx = rng.permutation(group.index.to_numpy())
        pool_indices[group_val] = shuffled_idx.tolist()

    test_idx_list = []
    train_idx_list = []

    pass_number = 1
    while True:
        # Check if every single disease has at least 5 cases left for this pass
        can_complete_pass = all(len(indices) >= 5 for indices in pool_indices.values())

        # If stop_on_first_exhaustion is True, halt algorithm to enforce perfect balance
        if stop_on_first_exhaustion and not can_complete_pass:
            print(f"Algorithm halted before Pass {pass_number}: A disease ran out of cases.")
            break

        groups_processed = 0

        for group_val, indices in pool_indices.items():
            if len(indices) >= 5:
                # Pop 1 random case for the test set
                test_idx_list.append(indices.pop(0))

                # Pop 4 random cases for the train set
                for _ in range(4):
                    train_idx_list.append(indices.pop(0))

                groups_processed += 1

        # Failsafe: if we aren't stopping on first exhaustion and all pools are empty
        if groups_processed == 0:
            print(f"Algorithm halted after Pass {pass_number - 1}: All diseases exhausted.")
            break

        pass_number += 1

    cap = 12  # max leftover cases per disease sent to train (tune this)
    for group_val, indices in pool_indices.items():
        train_idx_list.extend(indices[:cap])

    # Reconstruct the DataFrames using the collected indices
    train_df = df.loc[train_idx_list].copy().reset_index(drop=True)
    test_df = df.loc[test_idx_list].copy().reset_index(drop=True)

    return train_df, test_df

In [119]:
train_df, test_df = separate_naive_round_robin(df=all_cases_df)

Algorithm halted before Pass 3: A disease ran out of cases.


# **Upload dataset to HuggingFace**

In [ ]:
from huggingface_hub import login, whoami

# Retrieve token and set env var in one line
hf_token = get_secret("HF_TOKEN")

login(token=hf_token)
print(whoami())

In [ ]:
from datasets import Dataset, DatasetDict, List, Image, Value

def build_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    ds = ds.cast_column(
        "images",
        List({"image_path": Image(), "image_id": Value("string")}),
    )
    return ds

train_ds = build_hf_dataset(train_df)   # new names, don't overwrite train_df
test_ds = build_hf_dataset(test_df)

# Sanity check to see if images are encoded for real
print(train_ds.features)

In [ ]:
REPO_ID = "choisongjun/Tropical_Diseases_Repo_v2.0"
dataset_dict = DatasetDict({
    "train": build_hf_dataset(train_df.reset_index(drop=True)),
    "test": build_hf_dataset(test_df.reset_index(drop=True)),
})

In [ ]:
dataset_dict.push_to_hub(REPO_ID, token=hf_token)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")